In [1]:
#April 22, 2026
#Charlotte Meyer
#Data Seminar
#This code combines the cejst csv and the fractracker csv to create a pandas dataframe fit for logistic regression.
#It outputs a csv file titled "cejst_logistic_regression.csv"

In [2]:
#importing packages
import pandas as pd
import geopandas as gpd
from pygris import tracts

In [3]:
#setting pandas columns so I can see all of them
pd.set_option('display.max_columns', None)

#reading in data
cejst = pd.read_csv("CEJST_2.0-communities.csv", low_memory = False)
atlas = pd.read_csv('im3_open_source_data_center_atlas_v2026.02.09.csv')
#fractracker = pd.read_csv("U.S. Data Centers - FracTracker.csv")

In [4]:
#cleaning cejst

#renaming columns
cejst = cejst.rename(columns={'Census tract 2010 ID':'GEOID'})
cejst = cejst.rename(columns={"State/Territory":"State"})

cejst = cejst.rename(columns={'Percent White':'perc_white',
                              "Percent Black or African American alone":"perc_black",
                              "Percent American Indian / Alaska Native":"perc_native_american",
                              "Percent Asian":"perc_asian",
                              "Percent Native Hawaiian or Pacific":"perc_hawaiin_pacific",
                              "Adjusted percent of individuals below 200% Federal Poverty Line":"perc_poverty",
                              "Percent two or more races":"perc_mixed_race",
                              "PM2.5 in the air":"pm_air",
                              "Unemployment (percent)":"perc_unemployment",
                              "Proximity to hazardous waste sites":"hazard_waste_prox",
                              "Life expectancy (years)":"life_expectancy",
                              "Percent Hispanic or Latino":"perc_hispanic",
                              "Percent age over 64":"perc_elderly",
                              "Percent age under 10":"perc_children",
                              "Percentage of tract that is disadvantaged by area":"perc_disadvantaged",
                              "Total population":"population",
                              "Expected population loss rate (Natural Hazards Risk Index)":"loss_rate",
                              "Share of properties at risk of flood in 30 years":"flood_risk",
                              "Energy burden":"energy_burden",
                              "Diesel particulate matter exposure":"diesel_exposure",
                              "Traffic proximity and volume":"traffic_prox",
                              "DOT Travel Barriers Score (percentile)":"travel_barriers",
                              "Housing burden (percent)":"housing_burden",
                              "Median value ($) of owner-occupied housing units":"house_value",
                              "Share of the tract's land area that is covered by impervious surface or cropland as a percent (percentile)":"pavement_cropland",
                              "Share of homes with no kitchen or indoor plumbing (percentile)":"no_plumbing",
                              "Proximity to NPL (Superfund) sites":"superfund_prox",
                              "Proximity to Risk Management Plan (RMP) facilities":"risk_facilities_prox",
                              "Wastewater discharge (percentile)":"wastewater_discharge",
                              "Leaky underground storage tanks (percentile)":"leaky_storage_tanks",
                              "Current asthma among adults aged greater than or equal to 18 years (percentile)":"asthma",
                              "Diagnosed diabetes among adults aged greater than or equal to 18 years (percentile)":"diabetes",
                              "Coronary heart disease among adults aged greater than or equal to 18 years (percentile)":"heart_disease",
                              "Median household income as a percent of area median income":"household_income_compared",
                              "Linguistic isolation (percent) (percentile)":"linguistic_isolation",
                              "Percent individuals age 25 or over with less than high school degree":"no_hs_degree"})


In [5]:
# Read the shapefile
census_tracts = gpd.read_file("tl_2025_39_tract.zip")

#this is to remove the lakes
census_tracts = census_tracts[census_tracts['ALAND'] > 1_000_000]

#census_tracts.crs

In [6]:
#combining cejst and atlas into a dataset that I can use for logistic regression
#the following code is adapted from Claude

# --- STEP 1: Prepare atlas data ---
#selected_statuses = ["Operating", "Expanding", "Proposed"]
#fractracker_prop = fractracker[fractracker['Status'].isin(selected_statuses)].copy()
#fractracker_prop = fractracker[fractracker.Status == "Operating", "Expanding", "Proposed"].copy()
atlas_df = atlas.copy() 

# --- STEP 2: Download all US census tracts ---
states = [
    '01', '02', '04', '05', '06', '08', '09', '10', '11', '12', '13', '15', '16',
    '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29',
    '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42',
    '44', '45', '46', '47', '48', '49', '50', '51', '53', '54', '55', '56', '72'
]

all_tracts = []
for state in states:
    try:
        state_tracts = tracts(state=state, year=2021, cb=True)
        all_tracts.append(state_tracts)
    except Exception as e:
        print(f"Failed for {state}: {e}")

tract_geometries = gpd.GeoDataFrame(pd.concat(all_tracts, ignore_index=True))

# --- STEP 3: Prepare CEJST data with geometry ---
# Fix GEOID data types to match
cejst['GEOID'] = cejst['GEOID'].astype(str).str.zfill(11)  # Ensure 11 digits with leading zeros
tract_geometries['GEOID'] = tract_geometries['GEOID'].astype(str)

# Merge CEJST with tract geometries
cejst_with_geom = cejst.merge(
    tract_geometries[['GEOID', 'geometry']],
    on='GEOID',
    how='left'
)
cejst_with_geom = gpd.GeoDataFrame(cejst_with_geom, geometry='geometry', crs=tract_geometries.crs)

# --- STEP 4: Create point geometries for data centers ---
atlas_points = gpd.GeoDataFrame(
    atlas_df,
    geometry=gpd.points_from_xy(atlas_df['lon'], atlas_df['lat']),
    crs='EPSG:4326'
)

# Match CRS
atlas_points = atlas_points.to_crs(cejst_with_geom.crs)

# --- STEP 5: Spatial join ---
joined = gpd.sjoin(
    cejst_with_geom,
    atlas_points,
    how='left',
    predicate='contains'
)

# --- STEP 6: Create binary column - KEY FIX ---
# Check if index_right is NOT null (means a data center was found)
has_datacenter_df = joined.groupby(joined.index)['index_right'].apply(lambda x: x.notna().any())

# Map back to original cejst dataframe
cejst['has_datacenter'] = cejst.index.map(has_datacenter_df).fillna(False).astype(int)

# Check results
print(f"Tracts with data centers: {cejst['has_datacenter'].sum()}")
print(f"Tracts without data centers: {(cejst['has_datacenter'] == 0).sum()}")
print(f"Total tracts: {len(cejst)}")
print(f"Total data centers: {len(atlas_df)}")

Tracts with data centers: 403
Tracts without data centers: 73731
Total tracts: 74134
Total data centers: 1479


/var/folders/sl/dyq7sj0x2_z0dr2x82v1ln_80000gn/T/ipykernel_56879/1975089371.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cejst['has_datacenter'] = cejst.index.map(has_datacenter_df).fillna(False).astype(int)


In [7]:
#saving dataframe as csv

cejst.to_csv("cejst_logistic_regression.csv")

In [8]:
'''
fractracker results: 
Tracts with data centers: 410
Tracts without data centers: 73724
Total tracts: 74134
Total proposed/operating/expanding data centers: 1032
'''

'\nfractracker results: \nTracts with data centers: 410\nTracts without data centers: 73724\nTotal tracts: 74134\nTotal proposed/operating/expanding data centers: 1032\n'